# Online Retail II: EDA и прогноз дневной выручки

Датасет: транзакции британского интернет-магазина за 2009–2011 годы (UCI ML Repository).
Цель: показать полный цикл работы с «грязной» выгрузкой — профиль, очистка, EDA, признаки, `Pipeline` с `ColumnTransformer`, проверка утечек и сравнение вариантов препроцессинга.

Ноутбук читает полный файл `data/raw/online_retail_ii.csv`, если он есть, иначе выборку на 100 000 строк.

In [1]:
from pathlib import Path
import json

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data/raw"
PROCESSED = ROOT / "data/processed"
OUTPUTS = ROOT / "outputs"
FIGURES = OUTPUTS / "figures"
MODELS = ROOT / "models"
for folder in (PROCESSED, OUTPUTS, FIGURES, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 20)
print("project root:", ROOT)

project root: D:\ML\Projects\kwork\eda_model


In [2]:
full_path = RAW / "online_retail_ii.csv"
sample_path = RAW / "online_retail_ii_sample.csv"
path = full_path if full_path.exists() else sample_path

df = pd.read_csv(path, parse_dates=["InvoiceDate"])
print("file:", path.name)
print("shape:", df.shape)
df.head()

file: online_retail_ii_sample.csv
shape: (100000, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year
0,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010
1,489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010
2,489437,22143,CHRISTMAS CRAFT HEART DECORATIONS,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Year 2009-2010
3,489437,22130,PARTY CONE CHRISTMAS DECORATION,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom,Year 2009-2010
4,489437,21912,VINTAGE SNAKES & LADDERS,4,2009-12-01 09:08:00,3.75,15362.0,United Kingdom,Year 2009-2010


## 1. Профиль данных

Типы, пропуски, дубликаты и признаки «грязных» записей: отмены, отрицательные количества, нулевые цены.

In [3]:
df.info()

missing = df.isna().sum().to_frame("missing")
missing["share, %"] = (missing["missing"] / len(df) * 100).round(2)
missing[missing["missing"] > 0]

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      100000 non-null  str           
 1   StockCode    100000 non-null  str           
 2   Description  99610 non-null   str           
 3   Quantity     100000 non-null  int64         
 4   InvoiceDate  100000 non-null  datetime64[us]
 5   Price        100000 non-null  float64       
 6   Customer ID  77340 non-null   float64       
 7   Country      100000 non-null  str           
 8   Year         100000 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 6.9 MB


,missing,"share, %"
Description,390,0.39
Customer ID,22660,22.66


In [4]:
duplicates = df.duplicated().sum()
cancelled = df["Invoice"].astype(str).str.startswith("C").sum()
negative_qty = (df["Quantity"] < 0).sum()
zero_price = (df["Price"] <= 0).sum()

print(f"duplicates: {duplicates}")
print(f"cancelled invoices: {cancelled} ({cancelled / len(df) * 100:.1f}%)")
print(f"negative quantity: {negative_qty} ({negative_qty / len(df) * 100:.1f}%)")
print(f"price <= 0: {zero_price} ({zero_price / len(df) * 100:.1f}%)")

duplicates: 105
cancelled invoices: 1879 (1.9%)
negative quantity: 2216 (2.2%)
price <= 0: 565 (0.6%)


## 2. Очистка

Убираем отмены и возвраты, нулевые цены, строки без клиента и дубликаты. Каждый шаг считается, чтобы было видно, сколько данных теряется.

In [5]:
raw_rows = len(df)
work = df.copy()

step1 = work[~work["Invoice"].astype(str).str.startswith("C")]
step2 = step1[step1["Quantity"] > 0]
step3 = step2[step2["Price"] > 0]
step4 = step3.dropna(subset=["Customer ID"])
cleaned = step4.drop_duplicates().copy()

cleaned["Revenue"] = cleaned["Quantity"] * cleaned["Price"]

print(f"raw:       {raw_rows:>8,}")
print(f"no cancel: {len(step1):>8,}")
print(f"qty > 0:   {len(step2):>8,}")
print(f"price > 0: {len(step3):>8,}")
print(f"customer:  {len(step4):>8,}")
print(f"dedup:     {len(cleaned):>8,} ({len(cleaned) / raw_rows * 100:.1f}% от исходных)")

raw:        100,000
no cancel:   98,121
qty > 0:     97,784
price > 0:   97,556
customer:    75,514
dedup:       75,412 (75.4% от исходных)


In [6]:
cleaned.to_csv(PROCESSED / "retail_cleaned.csv", index=False)
print("saved:", PROCESSED / "retail_cleaned.csv")
cleaned.head()

saved: D:\ML\Projects\kwork\eda_model\data\processed\retail_cleaned.csv


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Year,Revenue
0,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010,59.50
1,489436,21754,HOME BUILDING BLOCK WORD,3,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010,17.85
2,489437,22143,CHRISTMAS CRAFT HEART DECORATIONS,6,2009-12-01 09:08:00,2.10,15362.0,United Kingdom,Year 2009-2010,12.60
3,489437,22130,PARTY CONE CHRISTMAS DECORATION,12,2009-12-01 09:08:00,0.85,15362.0,United Kingdom,Year 2009-2010,10.20
4,489437,21912,VINTAGE SNAKES & LADDERS,4,2009-12-01 09:08:00,3.75,15362.0,United Kingdom,Year 2009-2010,15.00


## 3. EDA

Выручка по месяцам, страны и товары-лидеры: сезонность и перекос по рынкам.

In [7]:
monthly = cleaned.set_index("InvoiceDate")["Revenue"].resample("ME").sum()

fig, ax = plt.subplots(figsize=(11, 4))
monthly.plot(ax=ax, marker="o", linewidth=1.5)
ax.set_title("Выручка по месяцам")
ax.set_xlabel("")
ax.set_ylabel("Выручка, £")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "revenue_by_month.png", dpi=150)
plt.show()

C:\Users\boroz\AppData\Local\Temp\ipykernel_9376\3794777171.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
top_countries = (
    cleaned.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(8)
)

fig, ax = plt.subplots(figsize=(8, 4))
top_countries.sort_values().plot.barh(ax=ax, color="#4C78A8")
ax.set_title("Выручка по странам, топ-8")
ax.set_xlabel("£")
fig.tight_layout()
fig.savefig(FIGURES / "top_countries.png", dpi=150)
plt.show()

top_countries.to_frame("revenue")

C:\Users\boroz\AppData\Local\Temp\ipykernel_9376\3898778233.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,revenue
Country,
United Kingdom,1333914.921
EIRE,56148.550
Netherlands,52032.570
Germany,38214.100
France,32616.350
Australia,17028.980
Spain,9459.570
Switzerland,9418.140


In [9]:
top_products = (
    cleaned.groupby("Description")["Revenue"].sum().sort_values(ascending=False).head(10)
)
top_products.to_frame("revenue")

,revenue
Description,
REGENCY CAKESTAND 3 TIER,25515.90
WHITE HANGING HEART T-LIGHT HOLDER,21611.60
ASSORTED COLOUR BIRD ORNAMENT,14967.13
JUMBO BAG RED RETROSPOT,11945.00
POSTAGE,10787.95
PARTY BUNTING,10227.60
Manual,8832.41
PAPER CHAIN KIT 50'S CHRISTMAS,7745.12
HEART OF WICKER SMALL,7641.50


## 4. Признаки

Агрегируем выручку по дням. Календарные признаки делаем категориальными (`dayofweek`, `month`) — для них пойдёт `OneHotEncoder`. Лаги считаются только по прошлым дням (`shift`), утечки будущего нет.

In [10]:
daily = cleaned.set_index("InvoiceDate")["Revenue"].resample("D").sum().to_frame()
daily = daily[daily["Revenue"] > 0].copy()

daily["dayofweek"] = daily.index.dayofweek.astype("category")
daily["month"] = daily.index.month.astype("category")
daily["is_weekend"] = (daily.index.dayofweek >= 5).astype(int)
daily["lag1"] = daily["Revenue"].shift(1)
daily["lag7"] = daily["Revenue"].shift(7)
daily["rolling7"] = daily["Revenue"].shift(1).rolling(7).mean()
daily = daily.dropna()

print("days:", len(daily), "period:", daily.index.min().date(), "-", daily.index.max().date())
daily.head()

days: 597 period: 2009-12-08 - 2011-12-09


,Revenue,dayofweek,month,is_weekend,lag1,lag7,rolling7
InvoiceDate,,,,,,,
2009-12-08,3909.84,1,12,0,2808.54,3986.26,3832.515714
2009-12-09,2814.09,2,12,0,3909.84,6314.53,3821.598571
2009-12-10,4116.91,3,12,0,2814.09,6903.99,3321.535714
2009-12-11,3084.10,4,12,0,4116.91,3149.91,2923.381429
2009-12-13,1740.23,6,12,1,3084.10,1124.66,2913.980000


## 5. Проверка утечек

Три вещи, которые ломают прогноз чаще всего: перемешанные даты, целевая переменная в признаках и препроцессор, обученный на всём датасете. Проверяем всё явно.

In [11]:
cat_features = ["dayofweek", "month"]
num_features = ["is_weekend", "lag1", "lag7", "rolling7"]
features = cat_features + num_features

X = daily[features]
y = daily["Revenue"]

split = int(len(daily) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# 1. Целевая переменная не должна быть среди признаков
assert "Revenue" not in features

# 2. Хронология: тест строго после train, без перемешивания
assert X_train.index.max() < X_test.index.min(), "train и test пересекаются по времени"

# 3. Лаг считается по прошлому дню, а не по текущему
assert daily["lag1"].iloc[1] == daily["Revenue"].iloc[0]
assert daily["lag7"].iloc[7] == daily["Revenue"].iloc[0]

# 4. В признаках нет пропусков, иначе модель молча выкинет строки
assert not X.isna().any().any()

print(f"train: {X_train.index.min().date()} — {X_train.index.max().date()} ({len(X_train)} дней)")
print(f"test:  {X_test.index.min().date()} — {X_test.index.max().date()} ({len(X_test)} дней)")
print("проверки утечки пройдены")

train: 2009-12-08 — 2011-07-21 (477 дней)
test:  2011-07-22 — 2011-12-09 (120 дней)
проверки утечки пройдены


## 6. Pipeline и варианты препроцессинга

`ColumnTransformer` разделяет колонки: категориальные кодируются `OneHotEncoder`, числовые масштабируются `StandardScaler`. Всё собрано в `Pipeline`, поэтому препроцессор обучается только на train — это и есть защита от утечки через статистики.

Сравниваем четыре варианта на одном holdout.

In [12]:
preprocess_scaled = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("num", StandardScaler(), num_features),
])

preprocess_raw = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("num", "passthrough", num_features),
])

preprocess_numeric = ColumnTransformer([
    ("num", StandardScaler(), num_features),
])

variants = {
    "ohe_scaler": Pipeline([("prep", preprocess_scaled), ("model", GradientBoostingRegressor(random_state=42))]),
    "ohe_raw": Pipeline([("prep", preprocess_raw), ("model", GradientBoostingRegressor(random_state=42))]),
    "numeric_scaler": Pipeline([("prep", preprocess_numeric), ("model", GradientBoostingRegressor(random_state=42))]),
}

def metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    r2 = r2_score(y_true, y_pred)
    return {
        "mae": round(float(mae), 2),
        "rmse": round(float(rmse), 2),
        "mape": round(mape, 2),
        "r2": round(float(r2), 3),
    }

results = {"baseline_lag1": metrics(y_test, X_test["lag1"])}
fitted = {}

for name, pipe in variants.items():
    pipe.fit(X_train[features], y_train)
    fitted[name] = pipe
    results[name] = metrics(y_test, pipe.predict(X_test[features]))

pd.DataFrame(results).T

,mae,rmse,mape,r2
baseline_lag1,1571.69,2035.76,59.37,-0.513
ohe_scaler,1187.07,1526.88,44.81,0.149
ohe_raw,1187.07,1526.88,44.81,0.149
numeric_scaler,1252.37,1602.01,43.69,0.063


In [13]:
# Препроцессор обучен только на train: средние StandardScaler совпадают со средними train
scaler = fitted["ohe_scaler"].named_steps["prep"].named_transformers_["num"]
train_means = X_train[num_features].mean().to_numpy()
print("scaler.mean_ == train means:", np.allclose(scaler.mean_, train_means))
print("scaler.mean_ == full means: ", np.allclose(scaler.mean_, X[num_features].mean().to_numpy()))

best_name = min(results, key=lambda k: results[k]["mae"])
best_model = fitted[best_name]
print("best by MAE:", best_name, results[best_name])

scaler.mean_ == train means: True
scaler.mean_ == full means:  False
best by MAE: ohe_scaler {'mae': 1187.07, 'rmse': 1526.88, 'mape': 44.81, 'r2': 0.149}


## 7. Результаты и сохранение

Лучший вариант сохраняем целиком — вместе с препроцессором, чтобы при инференсе не забыть про кодирование и масштабирование.

In [14]:
summary = {"variants": results, "best": best_name, "test_days": int(len(y_test))}
(OUTPUTS / "metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

pred = best_model.predict(X_test[features])
predictions = pd.DataFrame({"date": y_test.index, "actual": y_test.values, "predicted": pred})
predictions.to_csv(OUTPUTS / "predictions.csv", index=False)
joblib.dump(best_model, MODELS / "pipeline.pkl")
print("saved:", MODELS / "pipeline.pkl")

saved: D:\ML\Projects\kwork\eda_model\models\pipeline.pkl


In [15]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test.values, label="факт", linewidth=1.5)
ax.plot(y_test.index, pred, label="прогноз", linewidth=1.5, alpha=0.85)
ax.set_title(f"Дневная выручка: факт и прогноз ({best_name})")
ax.set_ylabel("£")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / "actual_vs_predicted.png", dpi=150)
plt.show()

C:\Users\boroz\AppData\Local\Temp\ipykernel_9376\186083811.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
prep = best_model.named_steps["prep"]
names = prep.get_feature_names_out()
importance = pd.Series(best_model.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
importance.head(10).sort_values().plot.barh(ax=ax, color="#72B7B2")
ax.set_title("Важность признаков")
fig.tight_layout()
fig.savefig(FIGURES / "feature_importance.png", dpi=150)
plt.show()

importance.head(10).to_frame("importance")

C:\Users\boroz\AppData\Local\Temp\ipykernel_9376\365933297.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,importance
num__rolling7,0.470986
num__lag1,0.177327
num__lag7,0.106921
cat__dayofweek_6,0.043231
num__is_weekend,0.036491
cat__month_12,0.036088
cat__dayofweek_3,0.026394
cat__dayofweek_4,0.021449
cat__month_11,0.017893
cat__month_9,0.010952


## 8. Выводы

- Утечек нет: тест идёт после train, `Revenue` не в признаках, лаги считаются по прошлым дням, препроцессор обучен только на train (средние `StandardScaler` совпадают со средними train, но не полного набора).
- `OneHotEncoder` + `StandardScaler` в `ColumnTransformer` даёт лучший MAE среди вариантов; масштабирование числовых признаков помогает бустингу слабее, чем кодирование календаря, но не вредит.
- Модель предсказывает дневную выручку по инерции (лаги и скользящее среднее) сильнее, чем по дню недели.
- Ограничения: один магазин, нет промо и праздников, горизонт — один день.

### Что дальше

- добавить праздники и промо-календарь;
- прогноз на неделю вперёд с рекурсивными лагами;
- отдельные модели по категориям товаров.
